# Notebook 02 — Algoritmo Genético

**Tech Challenge FIAP — Fase 2**

Otimização de hiperparâmetros do Random Forest via Algoritmo Genético.
São realizados **3 experimentos** com configurações diferentes para comparar convergência e resultado.

| Experimento | Pop | Gerações | Mutação | Cruzamento | Seed |
|-------------|-----|----------|---------|------------|------|
| 1 — Padrão       | 30 | 20 | 0.10 | 0.80 | 42 |
| 2 — Alta mutação | 30 | 20 | 0.30 | 0.80 | 43 |
| 3 — Pop. grande  | 60 | 30 | 0.10 | 0.90 | 44 |

**Referência (baseline GridSearch):** Random Forest F1-macro = 0.9017

## 0. Imports e configurações

In [1]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # backend não-interativo para execução headless
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipelines.preprocessing import build_preprocessed_splits
from src.models.baseline import build_pipelines
from src.genetic_algorithm.ga import run_genetic_algorithm
from src.evaluation.metrics import evaluate_model

EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Projeto raiz: {PROJECT_ROOT}")

Projeto raiz: /Users/igornatanael/workspace/tech-challenge-fiap-2


## 1. Dados e pipeline base

In [2]:
X_train, X_test, y_train, y_test, scaler = build_preprocessed_splits()
pipeline_base = build_pipelines()["random_forest"]

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")

PIPELINE DE PRE-PROCESSAMENTO
[load] Dataset carregado: 1014 linhas x 7 colunas
[outliers] Removidas 2 linha(s) com HeartRate < 30 bpm.
[age] Removidos 82 registro(s) com Age > 54 anos.
[dedup] Removidas 140 ocorrencia(s) acima do limite de 3 por grupo. Linhas restantes: 790
[features] BodyTemp convertida de Fahrenheit para Celsius.
[split] Treino: 632 linhas | Teste: 158 linhas (20% teste)
[scaler] StandardScaler aplicado (fit no treino, transform em ambos).
-------------------------------------------------------
Features finais (6): ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']
Distribuicao do alvo — treino:
  low risk (0): 277 (43.8%)
  mid risk (1): 198 (31.3%)
  high risk (2): 157 (24.8%)
Distribuicao do alvo — teste:
  low risk (0): 69 (43.7%)
  mid risk (1): 50 (31.6%)
  high risk (2): 39 (24.7%)
Treino: (632, 6) | Teste: (158, 6)


## 2. Resultados baseline (referência)

Carregamos os resultados do Notebook 01 para comparação ao final.

In [3]:
with open(EXPERIMENTS_DIR / "baseline_results.json") as f:
    baseline = json.load(f)

baseline_rf = baseline["random_forest"]
print(f"Baseline RF — F1-macro: {baseline_rf['f1_macro']} | ROC-AUC: {baseline_rf['roc_auc_macro_ovr']}")

Baseline RF — F1-macro: 0.9017 | ROC-AUC: 0.9813


## 3. Experimento 1 — Configuração Padrão

`pop=30 | gen=20 | mut=0.10 | cross=0.80`

In [4]:
result_exp1 = run_genetic_algorithm(
    model_name="random_forest",
    pipeline=build_pipelines()["random_forest"],
    X_train=X_train,
    y_train=y_train,
    population_size=30,
    generations=20,
    mutation_rate=0.10,
    crossover_rate=0.80,
    random_state=42,
)

print(f"\nMelhor indivíduo: {result_exp1['best_individual']}")
print(f"Melhor F1-macro CV: {result_exp1['best_fitness']:.4f}")


Algoritmo Genético — random_forest
Pop: 30 | Gerações: 20 | Mutação: 0.1 | Cruzamento: 0.8


[Gen   1/20] best=0.7857 | mean=0.7011 | global_best=0.7857


[Gen   2/20] best=0.7897 | mean=0.7161 | global_best=0.7897


[Gen   3/20] best=0.7897 | mean=0.7327 | global_best=0.7897


[Gen   4/20] best=0.7897 | mean=0.7579 | global_best=0.7897


...

[Gen  16/20] best=0.7963 | mean=0.7811 | global_best=0.7963


[Gen  17/20] best=0.7963 | mean=0.7843 | global_best=0.7963


[Gen  18/20] best=0.7963 | mean=0.7777 | global_best=0.7963


[Gen  19/20] best=0.7963 | mean=0.7771 | global_best=0.7963


[Gen  20/20] best=0.7963 | mean=0.7873 | global_best=0.7963

Melhor indivíduo: {'n_estimators': 73, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Melhor F1-macro:  0.7963
Tempo total:      450.2s

Melhor indivíduo: {'n_estimators': 73, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Melhor F1-macro CV: 0.7963


## 4. Experimento 2 — Alta Mutação

`pop=30 | gen=20 | mut=0.30 | cross=0.80`

In [5]:
result_exp2 = run_genetic_algorithm(
    model_name="random_forest",
    pipeline=build_pipelines()["random_forest"],
    X_train=X_train,
    y_train=y_train,
    population_size=30,
    generations=20,
    mutation_rate=0.30,
    crossover_rate=0.80,
    random_state=43,
)

print(f"\nMelhor indivíduo: {result_exp2['best_individual']}")
print(f"Melhor F1-macro CV: {result_exp2['best_fitness']:.4f}")


Algoritmo Genético — random_forest
Pop: 30 | Gerações: 20 | Mutação: 0.3 | Cruzamento: 0.8


[Gen   1/20] best=0.7714 | mean=0.7085 | global_best=0.7714


[Gen   2/20] best=0.7714 | mean=0.7243 | global_best=0.7714


[Gen   3/20] best=0.7714 | mean=0.7304 | global_best=0.7714


[Gen   4/20] best=0.7714 | mean=0.7363 | global_best=0.7714


...

[Gen  16/20] best=0.7897 | mean=0.7561 | global_best=0.7897


[Gen  17/20] best=0.7897 | mean=0.7515 | global_best=0.7897


[Gen  18/20] best=0.7921 | mean=0.7478 | global_best=0.7921


[Gen  19/20] best=0.7921 | mean=0.7388 | global_best=0.7921


[Gen  20/20] best=0.7921 | mean=0.7494 | global_best=0.7921

Melhor indivíduo: {'n_estimators': 91, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
Melhor F1-macro:  0.7921
Tempo total:      666.5s

Melhor indivíduo: {'n_estimators': 91, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
Melhor F1-macro CV: 0.7921


## 5. Experimento 3 — População Grande

`pop=60 | gen=30 | mut=0.10 | cross=0.90`

In [6]:
result_exp3 = run_genetic_algorithm(
    model_name="random_forest",
    pipeline=build_pipelines()["random_forest"],
    X_train=X_train,
    y_train=y_train,
    population_size=60,
    generations=30,
    mutation_rate=0.10,
    crossover_rate=0.90,
    random_state=44,
)

print(f"\nMelhor indivíduo: {result_exp3['best_individual']}")
print(f"Melhor F1-macro CV: {result_exp3['best_fitness']:.4f}")


Algoritmo Genético — random_forest
Pop: 60 | Gerações: 30 | Mutação: 0.1 | Cruzamento: 0.9


[Gen   1/30] best=0.7488 | mean=0.7001 | global_best=0.7488


[Gen   2/30] best=0.7635 | mean=0.7152 | global_best=0.7635


[Gen   3/30] best=0.7875 | mean=0.7303 | global_best=0.7875


[Gen   4/30] best=0.7875 | mean=0.7375 | global_best=0.7875


...

[Gen  26/30] best=0.7993 | mean=0.7837 | global_best=0.7993


[Gen  27/30] best=0.7993 | mean=0.7782 | global_best=0.7993


[Gen  28/30] best=0.7993 | mean=0.7778 | global_best=0.7993


[Gen  29/30] best=0.7993 | mean=0.7868 | global_best=0.7993


[Gen  30/30] best=0.7993 | mean=0.7775 | global_best=0.7993

Melhor indivíduo: {'n_estimators': 69, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
Melhor F1-macro:  0.7993
Tempo total:      1045.3s

Melhor indivíduo: {'n_estimators': 69, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
Melhor F1-macro CV: 0.7993


## 6. Curvas de convergência

In [7]:
experiments = [
    ("Exp 1 — Padrão",       result_exp1, "steelblue"),
    ("Exp 2 — Alta Mutação",  result_exp2, "darkorange"),
    ("Exp 3 — Pop. Grande",   result_exp3, "seagreen"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, result, color in experiments:
    gens = [h["generation"]   for h in result["history"]]
    best = [h["global_best"]  for h in result["history"]]
    mean = [h["mean_fitness"] for h in result["history"]]

    axes[0].plot(gens, best, label=label, color=color, linewidth=2)
    axes[1].plot(gens, mean, label=label, color=color, linewidth=2, linestyle="--")

# Linha de referência: baseline GridSearch
for ax in axes:
    ax.axhline(baseline_rf["f1_macro"], color="red", linestyle=":", linewidth=1.5,
               label=f"Baseline GridSearch ({baseline_rf['f1_macro']})")
    ax.set_xlabel("Geração")
    ax.set_ylabel("F1-macro")
    ax.legend(fontsize=8)
    ax.set_ylim(0.5, 1.0)

axes[0].set_title("Melhor fitness global por geração")
axes[1].set_title("Fitness médio da população por geração")

fig.savefig(FIGURES_DIR / "ga_convergence.png", dpi=150, bbox_inches="tight")
plt.close("all")

## 7. Avaliação no conjunto de teste — modelos otimizados

In [8]:
from sklearn.pipeline import Pipeline
from src.genetic_algorithm.encoding import decode_to_sklearn_params

resultados_ag = {}

for label, result in [("exp1_padrao", result_exp1),
                       ("exp2_alta_mutacao", result_exp2),
                       ("exp3_pop_grande", result_exp3)]:

    # Recria pipeline e aplica melhores hiperparâmetros
    pipe = build_pipelines()["random_forest"]
    params = decode_to_sklearn_params("random_forest", result["best_individual"])
    pipe.set_params(**params)
    pipe.fit(X_train, y_train)

    metricas = evaluate_model(
        nome=label,
        pipeline=pipe,
        X_test=X_test,
        y_test=y_test,
        plot_confusion=True,
        save_path=str(FIGURES_DIR / f"ga_confusion_{label}.png"),
    )
    metricas["best_individual"] = result["best_individual"]
    metricas["best_fitness_cv"] = result["best_fitness"]
    resultados_ag[label] = metricas


AVALIACAO — EXP1_PADRAO
              precision    recall  f1-score   support

    low risk       0.91      0.90      0.91        69
    mid risk       0.90      0.86      0.88        50
   high risk       0.90      0.97      0.94        39

    accuracy                           0.91       158
   macro avg       0.90      0.91      0.91       158
weighted avg       0.90      0.91      0.90       158

ROC-AUC macro OvR: 0.9811

AVALIACAO — EXP2_ALTA_MUTACAO
              precision    recall  f1-score   support

    low risk       0.91      0.88      0.90        69
    mid risk       0.86      0.88      0.87        50
   high risk       0.93      0.95      0.94        39

    accuracy                           0.90       158
   macro avg       0.90      0.90      0.90       158
weighted avg       0.90      0.90      0.90       158

ROC-AUC macro OvR: 0.9813



AVALIACAO — EXP3_POP_GRANDE
              precision    recall  f1-score   support

    low risk       0.91      0.88      0.90        69
    mid risk       0.88      0.86      0.87        50
   high risk       0.90      0.97      0.94        39

    accuracy                           0.90       158
   macro avg       0.90      0.91      0.90       158
weighted avg       0.90      0.90      0.90       158

ROC-AUC macro OvR: 0.9812


## 8. Comparativo: Baseline vs. AG

In [9]:
# Monta tabela comparativa
rows = []

rows.append({
    "modelo": "Baseline (GridSearch)",
    "f1_macro": baseline_rf["f1_macro"],
    "roc_auc": baseline_rf["roc_auc_macro_ovr"],
    "recall_high_risk": baseline_rf["recall_high_risk"],
    "accuracy": baseline_rf["accuracy"],
})

labels_display = {
    "exp1_padrao":       "AG Exp.1 — Padrão",
    "exp2_alta_mutacao": "AG Exp.2 — Alta Mutação",
    "exp3_pop_grande":   "AG Exp.3 — Pop. Grande",
}

for key, m in resultados_ag.items():
    rows.append({
        "modelo": labels_display[key],
        "f1_macro": m["f1_macro"],
        "roc_auc": m["roc_auc_macro_ovr"],
        "recall_high_risk": m["recall_high_risk"],
        "accuracy": m["accuracy"],
    })

df_comp = pd.DataFrame(rows).set_index("modelo")
print("Comparativo Final:")
df_comp

Comparativo Final:


,f1_macro,roc_auc,recall_high_risk,accuracy
modelo,,,,
Baseline (GridSearch),0.9017,0.9813,0.9487,0.8987
AG Exp.1 — Padrão,0.9070,0.9811,0.9744,0.9051
AG Exp.2 — Alta Mutação,0.9017,0.9813,0.9487,0.8987
AG Exp.3 — Pop. Grande,0.9013,0.9812,0.9744,0.8987


## 9. Salvar resultados dos experimentos

In [10]:
def save_experiment(path: Path, config: dict, result: dict, metricas: dict) -> None:
    data = {
        "name": config["name"],
        "description": config["description"],
        "config": result["config"],
        "results": {
            "best_individual": {k: str(v) for k, v in result["best_individual"].items()},
            "best_fitness_cv": result["best_fitness"],
            "test_metrics": {k: v for k, v in metricas.items() if k != "best_individual"},
            "elapsed_seconds": result["elapsed_seconds"],
        },
        "history": result["history"],
        "status": "completed",
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Salvo: {path.name}")


configs = [
    {"name": "Experimento 1 — Configuração Padrão",    "description": "Baseline do AG para comparação"},
    {"name": "Experimento 2 — Alta Mutação",            "description": "Taxa de mutação elevada para maior exploração"},
    {"name": "Experimento 3 — População Grande",        "description": "População maior para convergência mais precisa"},
]

for i, (result, key) in enumerate([
    (result_exp1, "exp1_padrao"),
    (result_exp2, "exp2_alta_mutacao"),
    (result_exp3, "exp3_pop_grande"),
]):
    path = EXPERIMENTS_DIR / f"experiment_0{i+1}_{'_'.join(configs[i]['name'].split()[2:]).lower().replace('ã', 'a').replace('ç', 'c')}.json"
    # usa os arquivos já existentes
    existing_files = ["experiment_01_default.json", "experiment_02_high_mutation.json", "experiment_03_large_population.json"]
    save_experiment(EXPERIMENTS_DIR / existing_files[i], configs[i], result, resultados_ag[key])

Salvo: experiment_01_default.json
Salvo: experiment_02_high_mutation.json
Salvo: experiment_03_large_population.json
